# Notebook 2 — AI Summariser v2

**Input:** `smart_news_raw.csv` (from Notebook 1 Apify version)  
**Model:** `facebook/bart-large-cnn` — open-source, ~1.6 GB, downloads once from HuggingFace  
**Output:** `smart_news_summarised.csv`

**What's new in v2:**
- Works with Apify pipeline columns (`description` instead of `full_news`)
- Preserves `section` and `age_hours` columns from Notebook 1
- Per-section summary audit at the end
- Skips articles that already have a summary (safe to re-run)

In [ ]:
# %pip install transformers torch pandas sentencepiece

In [ ]:
import pandas as pd
import torch
from transformers import pipeline
import time

print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
print(f'Device   : {"GPU" if torch.cuda.is_available() else "CPU (slower, still works)"}')

In [ ]:
# ═══════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════
INPUT_CSV        = 'smart_news_raw.csv'
OUTPUT_CSV       = 'smart_news_summarised.csv'
BART_MAX_TOKENS  = 130   # ~90-100 words before trimming
BART_MIN_TOKENS  = 30
INPUT_CHAR_LIMIT = 3000  # Truncate to stay within BART's 1024 token window
TARGET_WORDS     = 60

In [ ]:
# ═══════════════════════════════════════════════════════
# LOAD MODEL
# facebook/bart-large-cnn: https://huggingface.co/facebook/bart-large-cnn
# ~1.6 GB. Downloads once, cached locally on all future runs.
# ═══════════════════════════════════════════════════════
print('Loading facebook/bart-large-cnn...')
print('(First run downloads ~1.6 GB — normal)')

device     = 0 if torch.cuda.is_available() else -1
summariser = pipeline('summarization', model='facebook/bart-large-cnn', device=device)
print('Model loaded.')

In [ ]:
def trim_to_n_words(text, n=60):
    words = text.split()
    return ' '.join(words[:n]) + ('...' if len(words) > n else '')

def summarise_article(text):
    """
    Returns a <= 60-word summary.
    Input priority: description > title (fallback).
    """
    if not isinstance(text, str) or len(text.strip()) < 50:
        return 'Summary not available.'
    try:
        result = summariser(
            text[:INPUT_CHAR_LIMIT],
            max_length=BART_MAX_TOKENS,
            min_length=BART_MIN_TOKENS,
            do_sample=False
        )
        return trim_to_n_words(result[0]['summary_text'].strip(), TARGET_WORDS)
    except Exception as e:
        return f'Summary error: {str(e)[:60]}'

In [ ]:
# ═══════════════════════════════════════════════════════
# LOAD CSV
# ═══════════════════════════════════════════════════════
df = pd.read_csv(INPUT_CSV)
print(f'Loaded {len(df)} articles from {INPUT_CSV}')
print(f'Columns: {list(df.columns)}')

# Show section breakdown if available
if 'section' in df.columns:
    print(f'\nSection breakdown:')
    print(df['section'].value_counts().to_string())

In [ ]:
# ═══════════════════════════════════════════════════════
# SUMMARISE
# Text priority: description (Apify) > full_news (RSS) > title
# Skips rows that already have a summary (safe to re-run).
# ═══════════════════════════════════════════════════════
summaries = list(df.get('ai_summary', pd.Series([''] * len(df))).fillna(''))
total     = len(df)
skipped   = 0
start     = time.time()

for i, row in df.iterrows():
    # Skip if already summarised
    if isinstance(summaries[i], str) and len(summaries[i].strip()) > 10:
        skipped += 1
        continue

    # Pick best available text source
    text = ''
    for field in ['description', 'full_news', 'rss_summary', 'title']:
        candidate = row.get(field, '')
        if isinstance(candidate, str) and len(candidate.strip()) >= 50:
            text = candidate
            break

    summary = summarise_article(text)
    summaries[i] = summary

    elapsed   = time.time() - start
    done      = i + 1 - skipped
    avg_time  = elapsed / max(done, 1)
    remaining = avg_time * (total - i - 1)

    section_tag = f"[{row.get('section', '?')}]" if 'section' in df.columns else ''
    print(f'  [{i+1}/{total}] {section_tag} {str(row.get("title", ""))[:50]}...')
    print(f'         {summary[:85]}...')
    print(f'         Est. remaining: {remaining:.0f}s')

df['ai_summary'] = summaries
print(f'\nDone. {total - skipped} summarised, {skipped} already had summaries.')

In [ ]:
# ═══════════════════════════════════════════════════════
# REORDER COLUMNS & SAVE
# ═══════════════════════════════════════════════════════
col_order = [
    'website_name', 'section', 'title', 'ai_summary', 'description',
    'url', 'published_at', 'age_hours', 'scrape_time'
]
df = df[[c for c in col_order if c in df.columns]]
df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')

print(f'Saved {len(df)} articles to {OUTPUT_CSV}')
print(f'\nPreview:')
print(df[['section', 'title', 'ai_summary']].head(5).to_string(index=False))

In [ ]:
# ═══════════════════════════════════════════════════════
# WORD COUNT AUDIT
# ═══════════════════════════════════════════════════════
df['word_count'] = df['ai_summary'].apply(lambda x: len(str(x).split()))
over_limit       = df[df['word_count'] > 60]

print(f'Total articles     : {len(df)}')
print(f'Within 60 words    : {len(df) - len(over_limit)}')
print(f'Over 60 words      : {len(over_limit)}')
print(f'Avg summary length : {df["word_count"].mean():.1f} words')
print(f'Max summary length : {df["word_count"].max()} words')

In [ ]:
# ═══════════════════════════════════════════════════════
# PER-SECTION SUMMARY AUDIT
# ═══════════════════════════════════════════════════════
if 'section' in df.columns:
    section_order = ['Crime', 'Tech', 'Politics', 'Business', 'Science',
                     'Sport', 'Entertainment', 'Lifestyle', 'World']
    for section in section_order:
        subset = df[df['section'] == section]
        if subset.empty:
            continue
        print(f'\n{"─"*65}')
        print(f'[{section.upper()}] — {len(subset)} articles')
        for _, row in subset.head(3).iterrows():
            print(f"  SOURCE : {row.get('website_name', '?')}")
            print(f"  TITLE  : {row.get('title', '?')}")
            print(f"  SUMMARY: {row.get('ai_summary', '?')}")
            print()